# EDA — Dental Cavity Segmentation Dataset

Exploratory Data Analysis for the `teeth/patch_dataset` used to train the U-Net cavity segmentation model.

**Questions this notebook answers:**
1. Is the dataset structurally sound? (counts, image↔mask pairing, duplicates, leakage)
2. What do the images and masks actually look like?
3. How severe is the class imbalance? *(this justifies the focal + dice loss)*
4. How big / how many are the lesions per patch?
5. Do cavity pixels look different from background pixels in intensity?
6. Where do cavities occur spatially in the patches?
7. Do train and validation come from the same distribution?
8. How many patches come from the same source X-ray? *(leakage risk)*


In [ ]:
# ── 1. Setup ─────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, re, random
import numpy as np
import cv2
import matplotlib.pyplot as plt
from collections import Counter

random.seed(42); np.random.seed(42)

ROOT      = "/content/drive/MyDrive/teeth/patch_dataset"
TRAIN_IMG = f"{ROOT}/train/images"
TRAIN_MSK = f"{ROOT}/train/masks"
VAL_IMG   = f"{ROOT}/val/images"
VAL_MSK   = f"{ROOT}/val/masks"

# plotting style — consistent, no gridlines clutter
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, RED, SAGE = "#1E3A8F", "#C4453B", "#8FAE9C"


## 2. Dataset inventory & integrity

Before any statistics, verify the dataset is *structurally* trustworthy:
- **Counts** per split and the train/val ratio.
- **Pairing**: every image must have a mask with the identical filename (that's how the training loader joins them).
- **Split hygiene**: no filename should appear in both train and val.

If any of these fail, every downstream metric is suspect.


In [ ]:
# ── 2. Inventory & integrity ─────────────────────────────────────────
def pngs(d): return sorted(f for f in os.listdir(d) if f.lower().endswith(".png"))

tr_i, tr_m = pngs(TRAIN_IMG), pngs(TRAIN_MSK)
va_i, va_m = pngs(VAL_IMG),   pngs(VAL_MSK)

print(f"train : {len(tr_i):5d} images | {len(tr_m):5d} masks")
print(f"val   : {len(va_i):5d} images | {len(va_m):5d} masks")
print(f"split : {len(tr_i)/(len(tr_i)+len(va_i)):.1%} train / {len(va_i)/(len(tr_i)+len(va_i)):.1%} val")

miss_tr = set(tr_i) - set(tr_m); orph_tr = set(tr_m) - set(tr_i)
miss_va = set(va_i) - set(va_m); orph_va = set(va_m) - set(va_i)
print(f"\nimages missing a mask  — train: {len(miss_tr)}, val: {len(miss_va)}")
print(f"masks without an image — train: {len(orph_tr)}, val: {len(orph_va)}")

overlap = set(tr_i) & set(va_i)
print(f"\ntrain ∩ val filename overlap: {len(overlap)}  {'✅ clean' if not overlap else '❌ LEAKAGE!'}")


## 3. Image & mask properties

Check that every file is what the model expects: same resolution, grayscale-readable, masks effectively binary.
Masks are stored as PNGs, so anti-aliased edges can contain values between 0 and 255 — we count how many pixels are "in-between" to confirm that the `> 0.5` binarization step in training is necessary and harmless.


In [ ]:
# ── 3. Properties (sampled for speed) ────────────────────────────────
SAMPLE = 500
sample_files = random.sample(tr_i, min(SAMPLE, len(tr_i)))

shapes, img_means, img_stds, inbetween = set(), [], [], 0
for f in sample_files:
    img = cv2.imread(os.path.join(TRAIN_IMG, f), 0)
    msk = cv2.imread(os.path.join(TRAIN_MSK, f), 0)
    shapes.add(img.shape)
    img_means.append(img.mean()); img_stds.append(img.std())
    inbetween += int(((msk > 10) & (msk < 245)).sum())

print(f"distinct image shapes in sample : {shapes}")
print(f"mean intensity  : {np.mean(img_means):.1f} ± {np.std(img_means):.1f}  (0-255 scale)")
print(f"mean contrast (per-image std)   : {np.mean(img_stds):.1f}")
print(f"'in-between' mask pixels (10-245): {inbetween} across {len(sample_files)} masks"
      f" → binarization with >0.5 is {'required' if inbetween else 'a no-op'}")


## 4. Look at the data

Statistics lie less when you've seen the pictures. For a random handful of patches we show the X-ray, its mask, and a red overlay of the mask on the X-ray — the fastest way to sanity-check annotation quality (do the red blobs sit on plausible dark lesion areas?).


In [ ]:
# ── 4. Sample grid: image | mask | overlay ───────────────────────────
def overlay(img, msk):
    rgb = np.stack([img]*3, -1).astype(float)
    m = msk > 127
    rgb[m] = rgb[m]*0.35 + np.array([220, 60, 50])*0.65
    return rgb.astype(np.uint8)

show = random.sample(tr_i, 4)
fig, axes = plt.subplots(4, 3, figsize=(7.5, 10))
for r, f in enumerate(show):
    img = cv2.imread(os.path.join(TRAIN_IMG, f), 0)
    msk = cv2.imread(os.path.join(TRAIN_MSK, f), 0)
    for c, (pic, t) in enumerate([(img, "X-ray patch"), (msk, "mask"), (overlay(img, msk), "overlay")]):
        ax = axes[r, c]
        ax.imshow(pic, cmap=None if c == 2 else "gray"); ax.axis("off")
        if r == 0: ax.set_title(t, fontsize=10)
plt.tight_layout(); plt.show()


## 5. Class imbalance — the headline number

For every training mask we compute the **fraction of pixels that are cavity**. This is the single most important distribution in the whole project:

- If it's tiny (it is — ~3%), plain pixel accuracy and vanilla cross-entropy are both useless, which is *the* justification for the **focal + dice** loss.
- We also count **empty masks**: patches with no cavity at all. If there are none, the model never sees a healthy patch — a bias worth stating openly.


In [ ]:
# ── 5. Cavity-pixel ratio distribution ───────────────────────────────
def mask_ratios(msk_dir, files):
    out = []
    for f in files:
        m = cv2.imread(os.path.join(msk_dir, f), 0)
        out.append((m > 127).mean())
    return np.array(out)

tr_ratio = mask_ratios(TRAIN_MSK, tr_m)          # full pass on train
va_ratio = mask_ratios(VAL_MSK,  va_m)

for name, r in [("train", tr_ratio), ("val", va_ratio)]:
    print(f"{name:5s}: mean {r.mean():.2%} | median {np.median(r):.2%} | max {r.max():.2%}"
          f" | empty masks {(r == 0).sum()} / {len(r)} ({(r == 0).mean():.1%})")

fg = tr_ratio.mean()
print(f"\n→ background:cavity pixel ratio ≈ {1/fg - 1:.0f} : 1")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(tr_ratio*100, bins=60, color=NAVY, alpha=.85, label="train")
ax.hist(va_ratio*100, bins=60, color=RED,  alpha=.55, label="val")
ax.axvline(tr_ratio.mean()*100, color=RED, ls="--", lw=1,
           label=f"train mean = {tr_ratio.mean():.1%}")
ax.set_xlabel("% of patch pixels that are cavity"); ax.set_ylabel("patches")
ax.set_title("Class imbalance: cavity pixels per patch"); ax.legend()
plt.tight_layout(); plt.show()


## 6. Lesion structure — how many, how big?

The pixel ratio hides *shape* information. Using connected components on each mask:

- **lesions per patch** → does the post-processing assumption in `testing.py` (take the *largest* contour only) throw away real cavities?
- **lesion area in px** → informs the `area > 100` noise threshold, and shows how dice-hostile the task is (a few boundary pixels are a big share of a small lesion).


In [ ]:
# ── 6. Lesions per patch & lesion sizes ──────────────────────────────
n_lesions, lesion_areas = [], []
for f in tr_m:
    m = (cv2.imread(os.path.join(TRAIN_MSK, f), 0) > 127).astype(np.uint8)
    n, _, stats, _ = cv2.connectedComponentsWithStats(m, 8)
    comps = stats[1:, cv2.CC_STAT_AREA]           # drop background label 0
    n_lesions.append(len(comps)); lesion_areas += comps.tolist()

n_lesions = np.array(n_lesions); lesion_areas = np.array(lesion_areas)
print(f"lesions per patch : mean {n_lesions.mean():.2f} | "
      f"1 lesion: {(n_lesions==1).mean():.0%} | ≥2 lesions: {(n_lesions>=2).mean():.0%}")
print(f"lesion area (px)  : median {np.median(lesion_areas):.0f} | "
      f"p10 {np.percentile(lesion_areas,10):.0f} | p90 {np.percentile(lesion_areas,90):.0f}")
print(f"lesions smaller than the app's 100-px noise filter: {(lesion_areas<100).mean():.0%}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(*np.unique(n_lesions, return_counts=True), color=NAVY)
axes[0].set_xlabel("lesions in patch"); axes[0].set_ylabel("patches")
axes[0].set_title("Lesion count per patch")
axes[1].hist(lesion_areas, bins=60, color=SAGE)
axes[1].axvline(100, color=RED, ls="--", label="app noise filter (100 px)")
axes[1].set_xlabel("lesion area (px, at 128×128)"); axes[1].set_title("Lesion size distribution")
axes[1].legend()
plt.tight_layout(); plt.show()


## 7. Intensity: cavity vs background pixels

Is there even signal? Cavities (demineralized tissue) should appear **darker** than healthy enamel on radiographs. We pool pixel intensities under the mask vs outside it. Heavy overlap between the two curves = the task genuinely needs *context/shape* (a CNN), not a global threshold — a great interview talking point.


In [ ]:
# ── 7. Cavity vs background intensity ────────────────────────────────
cav_px, bg_px = [], []
for f in random.sample(tr_i, 400):
    img = cv2.imread(os.path.join(TRAIN_IMG, f), 0)
    m   = cv2.imread(os.path.join(TRAIN_MSK, f), 0) > 127
    cav_px.append(img[m]); bg_px.append(img[~m][::37])   # subsample background
cav_px = np.concatenate(cav_px); bg_px = np.concatenate(bg_px)

print(f"cavity px     : mean {cav_px.mean():.1f} ± {cav_px.std():.1f}")
print(f"background px : mean {bg_px.mean():.1f} ± {bg_px.std():.1f}")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(bg_px,  bins=64, density=True, color=NAVY, alpha=.7, label="background")
ax.hist(cav_px, bins=64, density=True, color=RED,  alpha=.7, label="cavity")
ax.set_xlabel("pixel intensity (0–255)"); ax.set_ylabel("density")
ax.set_title("Cavity pixels vs background — is a simple threshold enough?")
ax.legend(); plt.tight_layout(); plt.show()


## 8. Spatial prior — where do cavities sit in the patch?

Averaging all masks gives a heatmap of cavity locations. A strong center bias would mean the patches were cropped *centered on* lesions — the model could partly learn "look in the middle," which matters when you later slide the model across a full X-ray where cavities appear anywhere.


In [ ]:
# ── 8. Mean-mask heatmap ─────────────────────────────────────────────
acc = np.zeros((128, 128), np.float64)
for f in tr_m:
    m = cv2.imread(os.path.join(TRAIN_MSK, f), 0)
    if m.shape != (128, 128): m = cv2.resize(m, (128, 128))
    acc += (m > 127)
heat = acc / len(tr_m)

fig, ax = plt.subplots(figsize=(4.6, 4))
im = ax.imshow(heat, cmap="inferno")
ax.set_title("P(cavity) per pixel position"); ax.axis("off")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()
print(f"center 64×64 holds {heat[32:96,32:96].sum()/heat.sum():.0%} of all cavity mass")


## 9. Train vs validation — same distribution?

Compare per-image brightness/contrast and the cavity-ratio stats across splits. If val differs systematically from train, validation scores won't predict real performance.


In [ ]:
# ── 9. Distribution shift check ──────────────────────────────────────
def img_stats(d, files):
    mu, sd = [], []
    for f in files:
        im = cv2.imread(os.path.join(d, f), 0)
        mu.append(im.mean()); sd.append(im.std())
    return np.array(mu), np.array(sd)

tr_mu, tr_sd = img_stats(TRAIN_IMG, random.sample(tr_i, 600))
va_mu, va_sd = img_stats(VAL_IMG, va_i)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (a, b, t) in zip(axes, [(tr_mu, va_mu, "per-image mean intensity"),
                                (tr_sd, va_sd, "per-image contrast (std)")]):
    ax.hist(a, bins=40, density=True, color=NAVY, alpha=.7, label="train")
    ax.hist(b, bins=40, density=True, color=RED,  alpha=.6, label="val")
    ax.set_title(t); ax.legend()
plt.tight_layout(); plt.show()


## 10. Source-image grouping — the leakage question

Filenames look like `<source-image>.rf.<hash>_<patchidx>.png`: many patches share one **source X-ray**. We group by the filename stem to count patches per source and check whether any *source* contributes to both train and val.

> This is the honest caveat of the project: the split was done at **patch level**. If a source X-ray straddles both splits, validation is optimistic. Whatever this cell prints, you want to *know* it before the interviewer asks.


In [ ]:
# ── 10. Patches per source X-ray & cross-split sources ───────────────
stem = lambda f: re.sub(r"_\d+\.png$", "", f)     # strip trailing _<patchidx>.png

tr_src = Counter(stem(f) for f in tr_i)
va_src = Counter(stem(f) for f in va_i)

print(f"unique source X-rays — train: {len(tr_src)}, val: {len(va_src)}")
print(f"patches per source   — train: mean {np.mean(list(tr_src.values())):.1f}, "
      f"max {max(tr_src.values())}")

shared = set(tr_src) & set(va_src)
n_shared_patches = sum(va_src[s] for s in shared)
print(f"\nsource X-rays appearing in BOTH splits: {len(shared)}")
print(f"val patches whose source also appears in train: {n_shared_patches} "
      f"({n_shared_patches/len(va_i):.1%} of val)")
print("→ " + ("✅ split is effectively patient-level" if not shared else
              "⚠️ patch-level split: quote val metrics with this caveat, "
              "and propose group-wise splitting (GroupShuffleSplit on source id) as the fix"))

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(list(tr_src.values()), bins=40, color=NAVY)
ax.set_xlabel("patches extracted per source X-ray (train)"); ax.set_ylabel("source images")
ax.set_title("How many patches does one X-ray contribute?")
plt.tight_layout(); plt.show()


## 11. Summary — what to say in the interview

Run all cells, then read this back with the numbers filled in:

| Finding | Number (from above) | So what |
|---|---|---|
| Dataset size | 4,426 train / 730 val, 128×128 grayscale | patch-based design keeps lesion resolution |
| Pairing & split hygiene | 0 missing masks, 0 filename overlap | structurally sound |
| **Class imbalance** | ~3% cavity pixels (~33:1) | justifies focal + dice loss; pixel accuracy is meaningless |
| Empty masks | ~0% | model never sees healthy patches → add negatives (future work) |
| Lesions per patch | some patches have ≥2 | app's "largest contour only" can miss secondaries |
| Lesion sizes | many are small | small-structure segmentation is dice-hostile; explains why 0.8 dice is decent |
| Intensity overlap | cavity vs background distributions overlap | a threshold can't solve this; a CNN with context can |
| Spatial prior | check the heatmap | if center-biased, note it when moving to sliding-window inference |
| Train/val shift | compare the two histograms | supports (or challenges) trusting val metrics |
| **Source grouping** | see cell 10 output | the leakage caveat — own it before they ask |
